In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid

from src.config import DATA_DIR

mlflow.set_experiment("Prediction_Risque_Incendies")
print("MLflow configuré avec l'expérience : Prediction_Risque_Incendies")



2026/09/15 11:44:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/15 11:44:50 INFO mlflow.store.db.utils: Updating database tables
2026/09/15 11:44:56 INFO mlflow.tracking.fluent: Experiment with name 'Prediction_Risque_Incendies' does not exist. Creating a new experiment.


MLflow configuré avec l'expérience : Prediction_Risque_Incendies


In [2]:
# Chargement des données et métadonnées

dataset_path = DATA_DIR / 'data_processed' / 'incendies_features_v2.parquet'
df_dataset = pd.read_parquet(dataset_path)

metadata_path = dataset_path.with_suffix('.metadata.json')
dataset_metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}

# Définition des variables explicatives et de la cible
cols_to_exclude = [
    'code_insee', 'date_idx', 'annee', 'mois',
    'target_occurrence', 'nb_incendies', 'surface_brulee_ha',
    'target_log_surface'
]
X = [col for col in df_dataset.columns if col not in cols_to_exclude]
y = 'target_occurrence'

print(f"Dataset chargé : {df_dataset.shape} | Features sélectionnées ({len(X)})")

Dataset chargé : (2321280, 33) | Features sélectionnées (25)


In [3]:
# =============================================================================
# CONSTANTES DE SPLIT TEMPOREL (Source unique de vérité)
# =============================================================================
SPLIT_TRAIN_END  = 2022     # Train : <= 2022
SPLIT_VAL_YEAR   = 2023     # Validation (Grid Search & Early Stopping)
SPLIT_TEST_START = 2024     # Test final hors-temps : >= 2024

mask_train = df_dataset['annee'] <= SPLIT_TRAIN_END
mask_val   = df_dataset['annee'] == SPLIT_VAL_YEAR
mask_test  = df_dataset['annee'] >= SPLIT_TEST_START

X_train, y_train = df_dataset.loc[mask_train, X], df_dataset.loc[mask_train, y]
X_val,   y_val   = df_dataset.loc[mask_val, X],   df_dataset.loc[mask_val, y]
X_test,  y_test  = df_dataset.loc[mask_test, X],  df_dataset.loc[mask_test, y]

# Contrôle des volumes
def print_split_stats(name, y_sub):
    total = len(y_sub)
    positives = y_sub.sum()
    rate = (positives / total) * 100
    print(f"--- {name} : {total:,} lignes | Incendies : {positives:,} ({rate:.2f}%)".replace(",", " "))

print_split_stats(f"TRAIN (<= {SPLIT_TRAIN_END})", y_train)
print_split_stats(f"VAL   ({SPLIT_VAL_YEAR})", y_val)
print_split_stats(f"TEST  (>= {SPLIT_TEST_START})", y_test)

--- TRAIN (<= 2022) : 1 973 088 lignes | Incendies : 36 524 (1.85%)
--- VAL   (2023) : 116 064 lignes | Incendies : 2 282 (1.97%)
--- TEST  (>= 2024) : 232 128 lignes | Incendies : 3 289 (1.42%)


In [4]:
# Échantillonnage pour le Grid Search

# Sous-échantillonnage stratifié par année sur le Train uniquement
train_grid_parts = []
df_train_only = df_dataset.loc[mask_train].copy()

for year, year_data in df_train_only.groupby('annee', observed=True):
    positives = year_data[year_data[y] == 1]
    negatives = year_data[year_data[y] == 0]
    negative_sample = negatives.sample(
        n=min(len(negatives), len(positives) * 10),
        random_state=42 + int(year),
    )
    train_grid_parts.append(pd.concat([positives, negative_sample]))

train_grid = (
    pd.concat(train_grid_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

X_search_train = train_grid[X]
y_search_train = train_grid[y]
search_pos_weight = (len(y_search_train) - y_search_train.sum()) / y_search_train.sum()

# La validation du Grid Search pointe directement sur le jeu de validation global
X_search_val = X_val
y_search_val = y_val

print(f"Train complet : {len(y_train):,} lignes")
print(f"Train Grid Search (1:10) : {len(train_grid):,} lignes (scale_pos_weight: {search_pos_weight:.2f})")
print(f"Validation Grid Search ({SPLIT_VAL_YEAR}) : {len(y_search_val):,} lignes")

Train complet : 1,973,088 lignes
Train Grid Search (1:10) : 401,764 lignes (scale_pos_weight: 10.00)
Validation Grid Search (2023) : 116,064 lignes


In [5]:

# Exécution du Grid Search et sélection du meilleur modèle

model_grids = {
    'LightGBM': {
        'num_leaves': [31, 63],
        'max_depth': [-1, 8],
        'learning_rate': [0.05],
        'n_estimators': [300],
        'min_child_samples': [50],
    },
    'XGBoost': {
        'max_depth': [6, 8],
        'learning_rate': [0.05],
        'n_estimators': [300],
        'min_child_weight': [1, 5],
    },
}

search_results = []
for model_name, parameter_grid in model_grids.items():
    for parameters in ParameterGrid(parameter_grid):
        if model_name == 'LightGBM':
            estimator = lgb.LGBMClassifier(
                **parameters,
                scale_pos_weight=search_pos_weight,
                random_state=42,
                n_jobs=-1,
                verbosity=-1,
            )
        else:
            estimator = xgb.XGBClassifier(
                **parameters,
                scale_pos_weight=search_pos_weight,
                tree_method='hist',
                random_state=42,
                n_jobs=-1,
                eval_metric='logloss',
            )

        with mlflow.start_run(run_name=f'grid_{model_name}'):
            estimator.fit(X_search_train, y_search_train)
            val_predictions = estimator.predict_proba(X_search_val)[:, 1]
            metrics = {
                'pr_auc_validation': average_precision_score(y_search_val, val_predictions),
                'roc_auc_validation': roc_auc_score(y_search_val, val_predictions),
            }
            mlflow.log_params(parameters)
            mlflow.log_params({
                'model_name': model_name,
                'dataset_version': dataset_metadata.get('dataset_version', 'v2'),
                'dataset_sha256': dataset_metadata.get('sha256', ''),
                'grid_train_end': SPLIT_TRAIN_END,
                'validation_year': SPLIT_VAL_YEAR,
            })
            mlflow.log_metrics(metrics)

        search_results.append({
            'model': model_name,
            **parameters,
            **metrics,
        })

results_grid = pd.DataFrame(search_results).sort_values('pr_auc_validation', ascending=False).reset_index(drop=True)
results_grid

,model,learning_rate,max_depth,min_child_samples,n_estimators,num_leaves,pr_auc_validation,roc_auc_validation,min_child_weight
0,LightGBM,0.05,-1,50.0,300,31.0,0.154958,0.821142,NaN
1,LightGBM,0.05,8,50.0,300,31.0,0.154276,0.823155,NaN
2,XGBoost,0.05,6,NaN,300,NaN,0.151999,0.807937,1.0
3,XGBoost,0.05,6,NaN,300,NaN,0.151601,0.809137,5.0
4,LightGBM,0.05,-1,50.0,300,63.0,0.149720,0.818484,NaN
5,LightGBM,0.05,8,50.0,300,63.0,0.149564,0.820683,NaN
6,XGBoost,0.05,8,NaN,300,NaN,0.145114,0.801065,5.0
7,XGBoost,0.05,8,NaN,300,NaN,0.143502,0.799223,1.0


In [6]:
# Réentraînement final sur tout le Train et évaluation Test

best_row = results_grid.iloc[0]
best_model_name = best_row['model']
model_parameters = {}
for parameter_name in model_grids[best_model_name]:
    value = best_row.get(parameter_name)
    if pd.notna(value):
        expected_type = type(model_grids[best_model_name][parameter_name][0])
        model_parameters[parameter_name] = expected_type(value)

# Poids calculé sur l'ensemble Train complet (jusqu'à 2022 inclus)
final_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

if best_model_name == 'LightGBM':
    best_model = lgb.LGBMClassifier(
        **model_parameters,
        scale_pos_weight=final_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )
    mlflow_model_logger = mlflow.lightgbm.log_model
else:
    best_model = xgb.XGBClassifier(
        **model_parameters,
        scale_pos_weight=final_pos_weight,
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    )
    mlflow_model_logger = mlflow.xgboost.log_model

with mlflow.start_run(run_name=f'best_{best_model_name}'):
    # Entraînement sur tout le train disponible (<= SPLIT_TRAIN_END)
    best_model.fit(X_train, y_train)

    val_predictions = best_model.predict_proba(X_val)[:, 1]
    test_predictions = best_model.predict_proba(X_test)[:, 1]

    final_metrics = {
        f'pr_auc_{SPLIT_VAL_YEAR}': average_precision_score(y_val, val_predictions),
        f'roc_auc_{SPLIT_VAL_YEAR}': roc_auc_score(y_val, val_predictions),
        'pr_auc_test': average_precision_score(y_test, test_predictions),
        'roc_auc_test': roc_auc_score(y_test, test_predictions),
    }

    mlflow.log_params({
        **model_parameters,
        'model_name': best_model_name,
        'dataset_version': dataset_metadata.get('dataset_version', 'v2'),
        'dataset_sha256': dataset_metadata.get('sha256', ''),
        'train_end': SPLIT_TRAIN_END,
        'validation_year': SPLIT_VAL_YEAR,
        'test_start': SPLIT_TEST_START,
    })
    mlflow.log_metrics(final_metrics)
    mlflow_model_logger(best_model, name='model')

print(f"Meilleur modèle retenu : {best_model_name}")
print("Métriques finales :", final_metrics)

2026/09/15 11:46:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Meilleur modèle retenu : LightGBM
Métriques finales : {'pr_auc_2023': 0.15303073502914658, 'roc_auc_2023': 0.8136162706145742, 'pr_auc_test': 0.12424775493176246, 'roc_auc_test': 0.7719287570488254}


In [16]:
print("Tracking URI utilisé :", mlflow.get_tracking_uri())

Tracking URI utilisé : sqlite:////home/coule/Documents/projets/incendies/notebooks/mlflow.db
